# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. We follow a stepwise approach modeled on a best-practice template.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset overview
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity is referenced by its `@id`. This ensures precise engagement with the Croissant schema.

In [ ]:
# List all record sets by their @id
record_sets = []
for rs in metadata.recordSet:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name','')} description: {rs.get('description','')}")
    record_sets.append(rs['@id'])

# List the fields for each record set
for rs in metadata.recordSet:
    print(f"Fields in RecordSet {rs['@id']}: ")
    for field in rs.get('field', []):
        print(f"  Field @id: {field['@id']}, name: {field.get('name','')} dataType: {field.get('dataType','')}")

## 3. Data Extraction

Load data from one or more record sets into DataFrames. Use record set and field `@id`s noted above.


In [ ]:
# Choose record sets to load - add their @id below
record_sets_ids = record_sets
dataframes = {}

for record_set_id in record_sets_ids:
    # Load all records for the record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for RecordSet {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalizing, grouping. Use the chosen record set, referencing all entities by their `@id`.


In [ ]:
# Example: Pick a numeric field by its @id for EDA
# Suppose a record set has a numeric field with @id 'cr:field:age' and group field 'cr:field:sex'
# Replace these values with actual IDs if they differ.

# For illustration: get the first record set and try field exploration
if len(record_sets_ids) > 0:
    record_set_id = record_sets_ids[0]
    df = dataframes[record_set_id]
    
    # Find numeric fields
    numeric_field_id = None
    group_field_id = None

    # Identify numeric and grouping fields from metadata
    for rs in metadata.recordSet:
        if rs['@id'] == record_set_id:
            for field in rs.get('field', []):
                # Choose first Integer or Float field for demo
                dt = field.get('dataType', '')
                if dt in ('schema:Integer', 'schema:Float') and not numeric_field_id:
                    numeric_field_id = field['@id']
                # Choose first non-numeric (e.g. categorical) for grouping
                if dt in ('schema:Text', 'schema:Boolean') and not group_field_id:
                    group_field_id = field['@id']

    print(f"Selected numeric field for EDA: {numeric_field_id}")
    print(f"Selected group field for EDA: {group_field_id}")

    # Proceed only if found
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize the data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Histogram of numeric field and boxplot by group
if len(record_sets_ids) > 0:
    record_set_id = record_sets_ids[0]
    df = dataframes[record_set_id]

    # Use fields discovered in previous cell
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If group_field_id exists, plot boxplot
        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(8, 4))
            sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive clinical and molecular information on second primary colorectal cancer in cancer survivors.
- Using Croissant schema `@id` fields, we can reliably reference all entities for reproducibility.
- The EDA reveals outlier handling, normalization, and potential group-wise trends.
- This notebook can be extended for statistical modeling or deeper domain-specific analysis based on the FAIR² dataset.